# Лабораторная работа 1. Биграммная языковая модель

Цель работы — реализовать языковую модель из счётчиков: оценить $P(w_t\mid w_{t-1})$, сгенерировать текст, сравнить правдоподобие и перплексию. Для основной части используются только стандартная библиотека Python; готовые NLP-модели и библиотеки не применяются.

**Корпус:** четыре тома романа Л. Н. Толстого «Война и мир» из открытой коллекции [RusLit](https://github.com/d0rj/RusLit). В README коллекции указано, что тексты находятся в общественном достоянии. Корпус загружается в память при запуске и в репозиторий не записывается.

In [1]:
from math import exp, isinf, log
from random import Random
from urllib.parse import quote
from urllib.request import Request, urlopen
import re

class SimpleCounter:

    def __init__(self, iterable=None):
        self._data = {}
        if iterable is not None:
            self.update(iterable)

    def update(self, iterable):
        for item in iterable:
            self.add(item)

    def add(self, item, amount=1):
        self._data[item] = self._data.get(item, 0) + amount

    def __getitem__(self, item):
        return self._data.get(item, 0)

    def __setitem__(self, item, value):
        self._data[item] = value

    def get(self, item, default=0):
        return self._data.get(item, default)

    def items(self):
        return self._data.items()

    def keys(self):
        return self._data.keys()

    def values(self):
        return self._data.values()

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __bool__(self):
        return bool(self._data)

    def most_common(self, n=None):
        ordered = sorted(self._data.items(), key=lambda pair: pair[1], reverse=True)
        return ordered if n is None else ordered[:n]

SEED = 42
rng = Random(SEED)

## 1. Корпус и токенизация

Токен — последовательность русских букв по регулярному выражению `r"[а-яё]+"`. Текст переводится в нижний регистр. Знаки препинания, цифры и фрагменты на других языках отбрасываются: так модель предсказывает только слова, а не смешивает слова со служебными символами. Последние 10 % последовательности токенов откладываются до обучения и используются только для проверки.

In [2]:
BASE_URL = 'https://raw.githubusercontent.com/d0rj/RusLit/main/prose/Tolstoy/'
FILENAMES = [
    'Война и мир. Том 1.txt',
    'Война и мир. Том 2.txt',
    'Война и мир. Том 3.txt',
    'Война и мир. Том 4.txt',
]

def download_utf8(url):
    request = Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(request, timeout=60) as response:
        return response.read().decode('utf-8')

urls = [BASE_URL + quote(name) for name in FILENAMES]
corpus_text = '\n'.join(download_utf8(url) for url in urls)
print(f'Загружено символов: {len(corpus_text):,}')
print('Источник:', 'https://github.com/d0rj/RusLit')

Загружено символов: 3,501,795
Источник: https://github.com/d0rj/RusLit


In [3]:
TOKEN_RE = re.compile(r'[а-яё]+')

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

tokens = tokenize(corpus_text)
assert len(tokens) >= 100_000, 'Корпус должен содержать не менее 100 000 слов'

split_at = int(len(tokens) * 0.90)
train_tokens_raw = tokens[:split_at]
test_tokens_raw = tokens[split_at:]

word_counts = SimpleCounter(train_tokens_raw)
print(f'Всего слов в корпусе: {len(tokens):,}')
print(f'Обучающая часть: {len(train_tokens_raw):,} слов (90 %)')
print(f'Отложенная часть: {len(test_tokens_raw):,} слов (10 %)')
print(f'Размер словаря обучающей части: {len(word_counts):,}')
print('\n20 самых частых слов:')
for word, count in word_counts.most_common(20):
    print(f'{word:>15}  {count:>7,}')

Всего слов в корпусе: 502,295
Обучающая часть: 452,065 слов (90 %)
Отложенная часть: 50,230 слов (10 %)
Размер словаря обучающей части: 49,930

20 самых частых слов:
              и   21,417
              в   12,307
             не    8,434
            что    7,866
             он    7,381
             на    6,810
              с    6,334
            как    3,983
            его    3,969
              к    3,567
             то    3,514
              я    3,008
             но    2,663
            она    2,631
            это    2,449
           было    2,441
            все    2,120
             по    2,044
         сказал    1,965
             за    1,946
